In [1]:
# 4. Design your function here! 

# For this demo, we do a basic NDVI calculation.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df

In [2]:
# 3. Obtain the header information for the Earth Engine query and store it in an xarray object.
#    This code does not do a full query for the data (yet)! 

#    In this example, we are just querying some data from Landsat 8 imagery 
#    over a small watershed for demo purposes.
import ee

#ee.Authenticate()

ee.Initialize()

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

#Plumas = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")

# This is a dictionary that my code requires. You don't have to touch anything here for demo purposes
# (although it should work with anything, in theory. Feel free to change it if you'd like).
# These parameters get stored and are used to generate the header information needed when we do the full
# run.

vector_path = r"R:\SCRATCH\adrianom\code\big-raster-helper\geometry\Plumas_National_forest.geojson"

ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-05-08', '2018-05-10').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])

In [3]:
from robustraster import run
from robustraster.hooks import preview_dataset_hook

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': vector_path,
    'crs': 'EPSG:3310',
    'scale': 30
},
    user_function=compute_ndvi,
    tune_function=False,
    hooks={
        "after_dataset_loaded": preview_dataset_hook},
    export_kwargs={
        "flag": "GTiff", 
        "output_folder": 
        "rush123", 
        "vrt": True},
    dask_mode="custom",
    dask_kwargs={
        "n_workers": 16,
        "threads_per_worker": 1,
        "memory_limit": "2GB"
    }
)

Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Dask cluster started: <Client: 'tcp://127.0.0.1:61023' processes=16 threads=16, memory=29.80 GiB>
[robustraster] Dask workers authenticated to Earth Engine.

Dataset preview:
                     time              X             Y  SR_B4  SR_B5
0 2018-05-09 18:38:10.899 -145860.087241  151568.60228    NaN    NaN
1 2018-05-09 18:38:10.899 -145860.087241  151598.60228    NaN    NaN
2 2018-05-09 18:38:10.899 -145860.087241  151628.60228    NaN    NaN
3 2018-05-09 18:38:10.899 -145860.087241  151658.60228    NaN    NaN
4 2018-05-09 18:38:10.899 -145860.087241  151688.60228    NaN    NaN

User function output preview:
                     time              X             Y  SR_B4  SR_B5  ndvi
0 2018-05-09 18:38:10.899 -145860.087241  151568.60228    NaN    NaN   NaN
1 2018-05-09 18:38:10.899 -145860.087241  151598.60228    NaN    NaN   NaN
2 2018-05-09 18:38:10.899 -145860.087241  151628.60228    NaN    NaN   NaN
3 20

In [ ]:
from robustraster import run

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
        "geometry": vector_path,
        "crs": "EPSG:3310",
        "scale": 30
},
    user_function=compute_ndvi,
    user_function_args=(666,),
    user_function_kwargs={"number_to_add": 666},
    tune_function=False,
    export_kwargs={
        "flag": "GTiff", 
        "output_folder": "rush1234", 
        "vrt": True},
    dask_mode="custom",
    dask_kwargs={
        "n_workers": 16,
        "threads_per_worker": 1,
        "memory_limit": "2GB"
    }
)

In [ ]:
from robustraster import run

run(
    dataset=r"C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\demos\rush123\chunk_-334616969183363686_time_2018_05_09T18_38_10.899000000.tif",
    source="local",
    dataset_params=None,
    user_function=compute_ndvi,
    export_params={"flag": "GTiff", "output_folder": "rush123_local", "vrt": True},
    dask_mode="full"  # for single worker in local test mode
)

In [ ]:
from robustraster import run

run(
    dataset=ic,
    source="ee",
    dataset_params=dataset_params,
    user_function=compute_ndvi,
    export_params={"flag": "GCS", "gcs_credentials": r"C:\Users\Adriano Matos\Documents\credentials\cloud_storage\robust-raster-b9004f0d0608.json", "gcs_bucket": "dr_bucket_the_magnet_warrior2"},
    dask_mode="full"  # for single worker in local test mode
)